In [ ]:
import pandas as pd


df = pd.read_csv('final_food_system_all.csv')

print(df.head())
print(df.info())

   CountyFIPS    State          County  Pop2010  county_low_access_10_share  \
0        1001  Alabama  Autauga County    54571                    0.093803   
1        1003  Alabama  Baldwin County   182265                    0.012664   
2        1005  Alabama  Barbour County    27457                    0.169060   
3        1007  Alabama     Bibb County    22915                    0.015935   
4        1009  Alabama   Blount County    57322                    0.000000   

   county_low_income_low_access_10_share  GROCPTH16  SUPERCPTH16  CONVSPTH16  \
0                               0.042257   0.054271     0.018090    0.560802   
1                               0.004639   0.139753     0.033733    0.568650   
2                               0.088892   0.155195     0.038799    0.737177   
3                               0.004459   0.220916     0.044183    0.662749   
4                               0.000000   0.086863     0.017373    0.469059   

   SNAPSPTH17  ...  StructuralRisk_norm  Foo

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules


cols_to_use = [
    'FoodSystemType', 'PovertyGroup', 'PovertyRate', 'UnemploymentRate',
    'MedianIncome', 'FoodSystemFailureScore', 'DG_per_10k'
]

df_subset = df[cols_to_use].copy()


def bin_column(col, bins=3, labels=['Low', 'Medium', 'High']):
    return pd.qcut(col, q=bins, labels=labels, duplicates='drop')

df_subset['PovertyRate_bin'] = bin_column(df_subset['PovertyRate'])
df_subset['UnemploymentRate_bin'] = bin_column(df_subset['UnemploymentRate'])
df_subset['MedianIncome_bin'] = bin_column(df_subset['MedianIncome'])
df_subset['FoodSystemFailureScore_bin'] = bin_column(df_subset['FoodSystemFailureScore'])
df_subset['DG_per_10k_bin'] = bin_column(df_subset['DG_per_10k'])


df_apriori = pd.get_dummies(df_subset[[
    'FoodSystemType', 'PovertyGroup', 'PovertyRate_bin', 'UnemploymentRate_bin',
    'MedianIncome_bin', 'FoodSystemFailureScore_bin', 'DG_per_10k_bin'
]])


frequent_itemsets = apriori(df_apriori, min_support=0.1, use_colnames=True)


rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)


rules = rules.sort_values(by=['lift', 'confidence'], ascending=False)

print(frequent_itemsets.head())
print(rules.head(10))


rules.to_csv('apriori_association_rules.csv', index=False)

    support                       itemsets
0  0.912739  (FoodSystemType_High Failure)
1  0.200000            (PovertyGroup_High)
2  0.200000             (PovertyGroup_Low)
3  0.200000        (PovertyGroup_Moderate)
4  0.200000       (PovertyGroup_Very High)
                                           antecedents  \
506  (FoodSystemType_High Failure, PovertyGroup_Ver...   
507       (PovertyRate_bin_High, MedianIncome_bin_Low)   
407                           (PovertyGroup_Very High)   
402       (PovertyRate_bin_High, MedianIncome_bin_Low)   
518  (FoodSystemType_High Failure, PovertyGroup_Ver...   
521       (PovertyRate_bin_Low, MedianIncome_bin_High)   
513                           (PovertyGroup_Very High)   
500  (FoodSystemType_High Failure, PovertyRate_bin_...   
424                            (PovertyGroup_Very Low)   
421       (PovertyRate_bin_Low, MedianIncome_bin_High)   

                                           consequents  antecedent support  \
506       (PovertyRate_bi

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from itertools import combinations
from collections import Counter

def get_frequent_itemsets(df, min_support=0.1):
    n = len(df)
    min_count = n * min_support


    transactions = [set(row[row == 1].index) for _, row in df.iterrows()]


    item_counts = Counter()
    for t in transactions:
        item_counts.update(t)

    frequent_1 = {frozenset([item]): count/n for item, count in item_counts.items() if count >= min_count}


    item_pairs = Counter()
    freq_items_list = sorted([list(i)[0] for i in frequent_1])
    for t in transactions:

        relevant_items = [i for i in t if frozenset([i]) in frequent_1]
        for pair in combinations(sorted(relevant_items), 2):
            item_pairs[frozenset(pair)] += 1

    frequent_2 = {pair: count/n for pair, count in item_pairs.items() if count >= min_count}


    all_frequent = {**frequent_1, **frequent_2}


    item_triples = Counter()
    for t in transactions:
        relevant_items = [i for i in t if frozenset([i]) in frequent_1]
        for triple in combinations(sorted(relevant_items), 3):
             item_triples[frozenset(triple)] += 1

    frequent_3 = {triple: count/n for triple, count in item_triples.items() if count >= min_count}

    all_frequent.update(frequent_3)

    return all_frequent

def generate_rules(frequent_itemsets, min_confidence=0.5):
    rules = []
    for itemset, support in frequent_itemsets.items():
        if len(itemset) < 2:
            continue

        from itertools import chain, combinations
        def powerset(iterable):
            s = list(iterable)
            return chain.from_iterable(combinations(s, r) for r in range(1, len(s)))

        for antecedent in powerset(itemset):
            antecedent = frozenset(antecedent)
            consequent = itemset - antecedent
            if antecedent in frequent_itemsets:
                support_a = frequent_itemsets[antecedent]
                confidence = support / support_a
                if confidence >= min_confidence:
                    lift = confidence / frequent_itemsets[consequent] if consequent in frequent_itemsets else None
                    rules.append({
                        'antecedent': list(antecedent),
                        'consequent': list(consequent),
                        'support': support,
                        'confidence': confidence,
                        'lift': lift
                    })
    return pd.DataFrame(rules)


cols_to_use = [
    'FoodSystemType', 'PovertyGroup', 'PovertyRate', 'UnemploymentRate',
    'MedianIncome', 'FoodSystemFailureScore', 'DG_per_10k'
]
df_subset = df[cols_to_use].copy()

def bin_column(col, bins=3, labels=['Low', 'Medium', 'High']):

    name = col.name
    return pd.qcut(col, q=bins, labels=[f"{name}_{l}" for l in labels], duplicates='drop')

df_subset['PovertyRate_bin'] = bin_column(df_subset['PovertyRate'])
df_subset['UnemploymentRate_bin'] = bin_column(df_subset['UnemploymentRate'])
df_subset['MedianIncome_bin'] = bin_column(df_subset['MedianIncome'])
df_subset['FoodSystemFailureScore_bin'] = bin_column(df_subset['FoodSystemFailureScore'])
df_subset['DG_per_10k_bin'] = bin_column(df_subset['DG_per_10k'])

# One-hot encoding
cols_cat = ['FoodSystemType', 'PovertyGroup', 'PovertyRate_bin', 'UnemploymentRate_bin', 'MedianIncome_bin', 'FoodSystemFailureScore_bin', 'DG_per_10k_bin']
df_apriori = pd.get_dummies(df_subset[cols_cat])

# Run simplified Apriori
freq_itemsets = get_frequent_itemsets(df_apriori, min_support=0.15)
rules_df = generate_rules(freq_itemsets, min_confidence=0.6)

if not rules_df.empty:
    rules_df = rules_df.sort_values(by='lift', ascending=False)
    print(rules_df.head(20))
    rules_df.to_csv('apriori_rules.csv', index=False)
else:
    print("No rules found.")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

                                           antecedent  \
57                           [PovertyGroup_Very High]   
58  [PovertyRate_bin_PovertyRate_High, MedianIncom...   
74  [MedianIncome_bin_MedianIncome_High, PovertyRa...   
72                            [PovertyGroup_Very Low]   
32                            [PovertyGroup_Moderate]   
34  [FoodSystemType_High Failure, PovertyRate_bin_...   
7                [PovertyRate_bin_PovertyRate_Medium]   
33  [FoodSystemType_High Failure, PovertyGroup_Mod...   
6                             [PovertyGroup_Moderate]   
20                           [PovertyGroup_Very High]   
55  [FoodSystemType_High Failure, PovertyGroup_Ver...   
73  [MedianIncome_bin_MedianIncome_High, PovertyGr...   
31                            [PovertyGroup_Very Low]   
70  [FoodSystemType_High Failure, PovertyGroup_Ver...   
60  [MedianIncome_bin_MedianIncome_Low, PovertyGro...   
54                           [PovertyGroup_Very High]   
69                            [

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:

interesting_rules = rules_df[
    rules_df['antecedent'].apply(lambda x: any('FoodSystemFailureScore' in item or 'DG' in item for item in x)) |
    rules_df['consequent'].apply(lambda x: any('FoodSystemFailureScore' in item or 'DG' in item for item in x))
]

print("Rules involving FoodSystemFailureScore or DG:")
print(interesting_rules.head(15))


rules_df.to_csv('apriori_association_rules_full.csv', index=False)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Rules involving FoodSystemFailureScore or DG:
                                           antecedent  \
1   [FoodSystemFailureScore_bin_FoodSystemFailureS...   
36  [DG_per_10k_bin_DG_per_10k_High, FoodSystemFai...   
21  [FoodSystemFailureScore_bin_FoodSystemFailureS...   
43  [PovertyRate_bin_PovertyRate_High, FoodSystemF...   
42  [MedianIncome_bin_MedianIncome_Low, FoodSystem...   
0                  [DG_per_10k_bin_DG_per_10k_Medium]   
62  [MedianIncome_bin_MedianIncome_Low, DG_per_10k...   
63  [DG_per_10k_bin_DG_per_10k_High, UnemploymentR...   
8                    [DG_per_10k_bin_DG_per_10k_High]   
25                    [DG_per_10k_bin_DG_per_10k_Low]   
64  [DG_per_10k_bin_DG_per_10k_Low, UnemploymentRa...   
68  [MedianIncome_bin_MedianIncome_High, DG_per_10...   
24  [FoodSystemFailureScore_bin_FoodSystemFailureS...   

                       consequent   support  confidence      lift  
1   [FoodSystemType_High Failure]  0.333439    1.000000  1.095604  
36  [FoodSystemType

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

##### Observations

1) Socio-Economic Predictors of Poverty: There is a near-perfect association between a Very High Poverty Group classification and the combination of high poverty rates and low median income, validating the consistency of the group labels.

2) Food System Failure Risk: Counties with a High Food System Failure Score are almost certainly classified as having a High Failure Food System Type (Confidence: 1.0). This suggests the scoring model is highly consistent with the qualitative categorization.

3) The Dollar General Correlation: High densities of Dollar General stores (DG_per_10k_High) are strongly associated with High Failure food systems (93% confidence). This supports research suggesting these stores proliferate in areas with limited traditional grocery access.

4) Employment and Food Risk: High unemployment combined with high Dollar General density further strengthens the link to a High Failure food system classification (Lift: 1.03).

The rules highlight a specific shift in how Dollar General (DG) stores associate with food system health as their density increases:

Medium DG Density: Shows a confidence of approximately 96.5% for system failure.

High DG Density: While still highly associated with failure (93.2% confidence), these rules more frequently include UnemploymentRate_High.

This suggests that a Medium density of discount retail is already enough to signal a compromised food system. Further increases to High density are likely a secondary effect of labor market distress rather than a standalone driver of food access issues.